Install dependencies

In [ ]:
!pip install datasets sentence-transformers faiss-cpu transformers==4.46.3\
    keybert bertopic pypdf groq


Imports


In [ ]:
import pandas as pd
import faiss
import os

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
summarizer=pipeline("summarization",model="facebook/bart-large-cnn")
from keybert import KeyBERT



1. Data Loading & Cleaning

In [ ]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers",split='train')
print(dataset)


In [ ]:
df=pd.DataFrame(dataset)
df

In [ ]:
df.columns

In [ ]:
df=df[['title','abstract']]
df

In [ ]:
df.shape

In [ ]:
df=df.head(15000)

In [ ]:
df.isnull().sum()

In [ ]:
df["paper_text"]=df["title"]+" "+ df["abstract"]

In [ ]:
df[["paper_text"]]

In [ ]:
df[["paper_text"]].head()

In [ ]:
print(df["paper_text"].iloc[0])

2. Embeddings (Sentence Transformers)

In [ ]:
model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
print(type(model))

In [ ]:
df["paper_text"] = df["paper_text"].fillna('')
df["paper_text"]=df["paper_text"].str.replace("\n","  ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [ ]:
sample_text=df["paper_text"].iloc[0]
sample_text

In [ ]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:56]

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)

In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)

In [ ]:
embedding=model.encode(df["paper_text"].to_list(),batch_size=32,show_progress_bar=True)

In [ ]:
print(embedding.shape)
print(type(embedding))

In [ ]:
embedding.dtype

3. FAISS Index (cosine similarity via normalized inner product)

In [ ]:
embedding = embedding.copy()

In [ ]:
faiss.normalize_L2(embedding)

In [ ]:
index=faiss.IndexFlatIP(384)

In [ ]:
index.add(embedding)

In [ ]:
print(index.ntotal)

In [ ]:

# Define the exact path to your data folder
index_path = "../data/paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index")
    index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embedding)
    index = faiss.IndexFlatIP(384)
    index.add(embedding)

    # Ensure the directory exists before saving
    os.makedirs(os.path.dirname(index_path), exist_ok=True)

    # Save it using the new path variable
    faiss.write_index(index, index_path)
    print("FAISS index saved successfully to the data folder!")

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

In [ ]:
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

In [ ]:
print(df.iloc[10466],["title"])
print(df.iloc[11873],["title"])
print(df.iloc[10466],["abstract"])

In [ ]:
def search_paper(query,K=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,K)
  return D,I




In [ ]:
D,I=search_paper(query)
print(D)
print(I)

In [ ]:
def search_paper(query,K=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,K)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score",score)
    print("Paper Title",df.iloc[idx]["title"])
    print("Paper Abstract",df.iloc[idx]["abstract"][:500])
    print()
  return D,I

In [ ]:
search_paper("deep learning for medical image analysis")

4. Abstract Summarization (BART)

In [ ]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

In [ ]:
for score,idx in zip(D[0],I[0]):
    print("Similarity Score",score)
    print("Paper Title",df.iloc[idx]["title"])
    print("Paper Abstract",df.iloc[idx]["abstract"][:500])

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print( )

In [ ]:
def search_and_summarizer(query,K=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,K)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score",score)
    print("Paper Title",df.iloc[idx]["title"])
    print("Paper Abstract",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print(summary)
    print()

In [ ]:
search_and_summarizer("deep learning for medical image analysis",K=5)

5. Key:word Extraction (KeyBERT)

In [ ]:
kw_model = KeyBERT()

In [ ]:
type(kw_model)

In [ ]:
text=df.iloc[10466]["abstract"]
keywords = kw_model.extract_keywords(text)
print(keywords)
print(type(keywords))
print(type(keywords[0]))

In [ ]:
finalkeyword=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None)
finalkeyword

Feature 1 — Named Entity Recognition (NER)

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
def extract_entities(text):

    if text is None:
        return []

    doc = nlp(text)

    entities = []

    for ent in doc.ents:

        entities.append({
            "text": ent.text,
            "label": ent.label_
        })

    return entities

In [ ]:
from tqdm import tqdm

tqdm.pandas()

df["entities"] = df["abstract"].progress_apply(extract_entities)

In [ ]:
df["entities"].iloc[0]

In [ ]:
def search_and_summarizer(query, K=5):

    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, K)

    for score, idx in zip(D[0], I[0]):

        print("="*80)

        print("Similarity Score :", round(score*100,2),"%")

        print()

        print("Paper Title")

        print(df.iloc[idx]["title"])

        print()

        print("Named Entities")

        entities = df.iloc[idx]["entities"]

        if len(entities)==0:

            print("No entities found")

        else:

            for entity in entities:

                print(f"{entity['text']} ({entity['label']})")

        print()

        print("Abstract")

        print(df.iloc[idx]["abstract"][:600])

        print()

        print("Summary")

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print(summary[0]["summary_text"])

        print()

    return D, I

In [ ]:
search_and_summarizer(
    "Deep Learning for Medical Image Analysis"
)

Feature 2: Keywords with Entity Type

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

CANDIDATE_LABELS = [
    "Programming Language", "Framework", "Library",
    "Algorithm", "Model Architecture", "Dataset",
    "Evaluation Metric", "Research Concept or Technique",
    "Application Domain", "Data Type or Modality",
    "Task or Problem Type", "Image Processing Task",
]

# Curated dictionary - expand as you find more common ML/CS terms
ENTITY_DICT = {
    "python": "Programming Language", "java": "Programming Language", "c++": "Programming Language",
    "fastapi": "Framework", "django": "Framework", "flask": "Framework", "react": "Framework",
    "pytorch": "Library", "tensorflow": "Library", "scikit-learn": "Library", "keras": "Library",
    "bert": "Model Architecture", "gpt": "Model Architecture", "resnet": "Model Architecture",
    "transformer": "Model Architecture", "lstm": "Model Architecture", "cnn": "Model Architecture",
    "cnns": "Model Architecture", "rnn": "Model Architecture", "alexnet": "Model Architecture",
    "convolutional neural network": "Model Architecture", "convolutional neural networks": "Model Architecture",
    "bleu": "Evaluation Metric", "f1 score": "Evaluation Metric", "accuracy": "Evaluation Metric",
    "precision": "Evaluation Metric", "recall": "Evaluation Metric",
    "imagenet": "Dataset", "coco": "Dataset", "mnist": "Dataset",
    "gradient descent": "Algorithm", "backpropagation": "Algorithm", "random forest": "Algorithm",
}



In [ ]:
def get_entity_type(keyword, confidence_threshold=0.25):
    keyword_clean = keyword.lower().strip()

    # Step 1: dictionary lookup (fast, precise)
    if keyword_clean in ENTITY_DICT:
        return ENTITY_DICT[keyword_clean], 1.0

    # Step 2: fallback to zero-shot classification
    result = zero_shot(keyword, CANDIDATE_LABELS, hypothesis_template="This term refers to a {}.")
    label, score = result["labels"][0], result["scores"][0]

    if score < confidence_threshold:
        return "Unclassified", score
    return label, score

test_keywords = ["python", "fastapi", "bert", "gradient boosting", "attention mechanism"]
for kw in test_keywords:
    entity_type, confidence = get_entity_type(kw)
    print(f"{kw:25s} -> {entity_type:25s} (confidence: {confidence:.2f})")

Full pipeline: search + summarize + keywords + entity types

In [ ]:
def summarize_abstract(abstract_text, max_length=120, min_length=40, do_sample=False):
    # Ensure abstract_text is a string, handle potential None or non-string types
    if not isinstance(abstract_text, str):
        return "" # Return empty string for invalid input

    # The summarizer expects a list of strings
    # The global `summarizer` pipeline is used here
    summary = summarizer([abstract_text], max_length=max_length, min_length=min_length, do_sample=do_sample)
    return summary[0]['summary_text'] if summary else ""

def extract_keywords(text, K=5):
    # The global `kw_model` is used here
    if not isinstance(text, str):
        return []
    return kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None, top_n=K)


def search_and_summarize(query, K=5):
    """Returns a list of dicts: score, title, abstract, summary, keywords (with entity types)."""
    # Call the existing search_paper function which returns D (distances) and I (indices)
    D, I = search_paper(query, K=K)

    results_list = []
    for score, idx in zip(D[0], I[0]):
        abstract_text = df.iloc[idx]["abstract"]
        title_text = df.iloc[idx]["title"]
        results_list.append({
            "score": score,
            "title": title_text,
            "abstract": abstract_text,
            "index": idx
        })

    for r in results_list:
        r["summary"] = summarize_abstract(r["abstract"])

        keywords = extract_keywords(r["abstract"], K=5)
        r["keywords"] = [
            {"keyword": kw, "entity_type": get_entity_type(kw)[0], "confidence": round(get_entity_type(kw)[1], 2)}
            for kw, _ in keywords
        ]
    return results_list

results = search_and_summarize("deep learning for medical image analysis")
for r in results:
    print(f"Similarity Score: {r['score']:.4f}")
    print("Paper Title:", r["title"])
    print("Summary:", r["summary"])
    print("Keywords:")
    for k in r["keywords"]:
        print(f"  - {k['keyword']} -> {k['entity_type']} (confidence: {k['confidence']:.2f})")
    print()